In [0]:
%sql
SELECT * FROM superstore_limpio LIMIT 10;

Row_ID,Order_ID,Order_Date,Ship_Date,Ship_Mode,Customer_ID,Customer_Name,Segment,Country,City,State,Postal_Code,Region,Product_ID,Category,Sub_Category,Product_Name,Sales,Quantity,Discount,Profit
26,CA-2017-121755,2017-01-16,2017-01-20,Second Class,EH-13945,Eric Hoffmann,Consumer,United States,Los Angeles,California,90049,West,OFF-BI-10001634,Office Supplies,Binders,Wilson Jones Active Use Binders,11.648,2,0.2,4.2224
29,US-2016-150630,2016-09-17,2016-09-21,Standard Class,TB-21520,Tracy Blumstein,Consumer,United States,Philadelphia,Pennsylvania,19140,East,OFF-BI-10000474,Office Supplies,Binders,Avery Recycled Flexi-View Covers for Binding Systems,9.618,2,0.7,-7.0532
30,US-2016-150630,2016-09-17,2016-09-21,Standard Class,TB-21520,Tracy Blumstein,Consumer,United States,Philadelphia,Pennsylvania,19140,East,FUR-FU-10004848,Furniture,Furnishings,"Howard Miller 13-3/4"" Diameter Brushed Chrome Round Wall Clock",124.2,3,0.2,15.525
57,CA-2017-111682,2017-06-17,2017-06-18,First Class,TB-21055,Ted Butterfield,Consumer,United States,Troy,New York,12180,East,OFF-PA-10001569,Office Supplies,Paper,Xerox 232,32.4,5,0.0,15.552
74,US-2016-134026,2016-04-26,2016-05-02,Standard Class,JE-15745,Joel Eaton,Consumer,United States,Memphis,Tennessee,38109,South,FUR-FU-10003708,Furniture,Furnishings,"Tenex Traditional Chairmats for Medium Pile Carpet, Standard Lip, 36"" x 48""",97.04,2,0.2,1.213
75,US-2016-134026,2016-04-26,2016-05-02,Standard Class,JE-15745,Joel Eaton,Consumer,United States,Memphis,Tennessee,38109,South,OFF-ST-10004123,Office Supplies,Storage,Safco Industrial Wire Shelving System,72.784,1,0.2,-18.196
82,CA-2015-139451,2015-10-12,2015-10-16,Standard Class,DN-13690,Duane Noonan,Consumer,United States,San Francisco,California,94122,West,OFF-AR-10002053,Office Supplies,Art,"Premium Writing Pencils, Soft, #2 by Central Association for the Blind",14.9,5,0.0,4.172
92,CA-2017-109806,2017-09-17,2017-09-22,Standard Class,JS-15685,Jim Sink,Corporate,United States,Los Angeles,California,90036,West,OFF-PA-10000304,Office Supplies,Paper,Xerox 1995,6.48,1,0.0,3.1104
118,CA-2016-110457,2016-03-02,2016-03-06,Standard Class,DK-13090,Dave Kipp,Consumer,United States,Seattle,Washington,98103,West,FUR-TA-10001768,Furniture,Tables,Hon Racetrack Conference Tables,787.53,3,0.0,165.3813
128,US-2018-107272,2018-11-05,2018-11-12,Standard Class,TS-21610,Troy Staebel,Consumer,United States,Phoenix,Arizona,85023,West,OFF-ST-10002974,Office Supplies,Storage,"Trav-L-File Heavy-Duty Shuttle II, Black",243.992,7,0.2,30.499


1. Ranking de los 10 mejores clientes por gasto total (usa DENSE_RANK)

In [0]:
%sql
SELECT Customer_Name,
       ROUND(SUM(Sales),2) AS Ventas_Totales,
       DENSE_RANK() OVER (ORDER BY SUM(Sales) DESC) AS Ranking
FROM superstore_limpio
GROUP BY Customer_Name
ORDER BY Ranking
LIMIT 10;

Customer_Name,Ventas_Totales,Ranking
Sean Miller,25043.05,1
Tamara Chand,19052.22,2
Raymond Buch,15117.34,3
Tom Ashbrook,14595.62,4
Adrian Barton,14473.57,5
Ken Lonsdale,14175.23,6
Sanjit Chand,14142.33,7
Hunter Lopez,12873.3,8
Sanjit Engle,12209.44,9
Christopher Conant,12129.07,10


Databricks visualization. Run in Databricks to view.

2. Clasificación de pedidos por rentabilidad (usa CASE WHEN)
sql

In [0]:
%sql
SELECT Category,
       CASE WHEN Profit > 0 THEN 'Rentable' ELSE 'Pérdida' END AS Estado_Rentabilidad,
       COUNT(*) AS Cantidad_Pedidos,
       ROUND(SUM(Profit),2) AS Ganancia_Total
FROM superstore_limpio
GROUP BY Category, CASE WHEN Profit > 0 THEN 'Rentable' ELSE 'Pérdida' END
ORDER BY Category, Estado_Rentabilidad;

Category,Estado_Rentabilidad,Cantidad_Pedidos,Ganancia_Total
Furniture,Pérdida,747,-60936.11
Furniture,Rentable,1372,78195.45
Office Supplies,Pérdida,915,-56615.26
Office Supplies,Rentable,5105,178500.29
Technology,Pérdida,274,-38579.92
Technology,Rentable,1570,183587.58


Databricks visualization. Run in Databricks to view.

3. Evolución mensual de ventas con variación respecto al mes anterior (usa LAG())


In [0]:
%sql
WITH ventas_mensuales AS (
  SELECT DATE_TRUNC('month', Order_Date) AS Mes,
         ROUND(SUM(Sales),2) AS Ventas_Mes
  FROM superstore_limpio
  GROUP BY DATE_TRUNC('month', Order_Date)
)
SELECT Mes, Ventas_Mes,
       ROUND(Ventas_Mes - LAG(Ventas_Mes) OVER (ORDER BY Mes), 2) AS Variacion_vs_Mes_Anterior
FROM ventas_mensuales
ORDER BY Mes;

Mes,Ventas_Mes,Variacion_vs_Mes_Anterior
2015-01-01T00:00:00.000Z,14236.9,null
2015-02-01T00:00:00.000Z,4519.89,-9717.01
2015-03-01T00:00:00.000Z,55691.01,51171.12
2015-04-01T00:00:00.000Z,28295.34,-27395.67
2015-05-01T00:00:00.000Z,23648.29,-4647.05
2015-06-01T00:00:00.000Z,34595.13,10946.84
2015-07-01T00:00:00.000Z,33946.39,-648.74
2015-08-01T00:00:00.000Z,27909.47,-6036.92
2015-09-01T00:00:00.000Z,81777.35,53867.88
2015-10-01T00:00:00.000Z,31453.39,-50323.96


Databricks visualization. Run in Databricks to view.

4. Sub-categorías con descuento promedio alto y su rentabilidad (subconsulta + HAVING)


In [0]:
%sql
SELECT Sub_Category,
       ROUND(AVG(Discount),3) AS Descuento_Promedio,
       ROUND(AVG(Profit),2) AS Ganancia_Promedio
FROM superstore_limpio
GROUP BY Sub_Category
HAVING AVG(Discount) > 0.15
ORDER BY Descuento_Promedio DESC;

Sub_Category,Descuento_Promedio,Ganancia_Promedio
Binders,0.372,19.84
Machines,0.306,29.43
Tables,0.261,-55.57
Bookcases,0.212,-19.76
Chairs,0.17,42.88
Appliances,0.167,38.68
Copiers,0.162,817.91
Phones,0.155,49.75


Databricks visualization. Run in Databricks to view.

5. Clientes que compran en más de una categoría (subconsulta con COUNT DISTINCT)

In [0]:
%sql
SELECT Customer_Name, COUNT(DISTINCT Category) AS Categorias_Distintas
FROM superstore_limpio
GROUP BY Customer_Name
HAVING COUNT(DISTINCT Category) = 3
ORDER BY Customer_Name;

Customer_Name,Categorias_Distintas
Aaron Bergman,3
Aaron Hawkins,3
Aaron Smayling,3
Adam Bellavance,3
Adam Hart,3
Adam Shillingsburg,3
Adrian Barton,3
Adrian Hane,3
Aimee Bixby,3
Alan Barnes,3


Databricks visualization. Run in Databricks to view.

6. Top 5 y bottom 5 estados por rentabilidad (ranking doble)


In [0]:
%sql
SELECT State, ROUND(SUM(Profit),2) AS Ganancia_Total,
       RANK() OVER (ORDER BY SUM(Profit) DESC) AS Ranking_Ganancia
FROM superstore_limpio
GROUP BY State
ORDER BY Ranking_Ganancia
LIMIT 5;

State,Ganancia_Total,Ranking_Ganancia
California,76381.39,1
New York,74038.55,2
Washington,33402.65,3
Michigan,24463.19,4
Virginia,18597.95,5


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT State, ROUND(SUM(Profit),2) AS Ganancia_Total,
       RANK() OVER (ORDER BY SUM(Profit) ASC) AS Ranking_Ganancia
FROM superstore_limpio
GROUP BY State
ORDER BY Ranking_Ganancia
LIMIT 5;

State,Ganancia_Total,Ranking_Ganancia
Texas,-25729.36,1
Ohio,-16971.38,2
Pennsylvania,-15559.96,3
Illinois,-12607.89,4
North Carolina,-7490.91,5


Databricks visualization. Run in Databricks to view.